In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
encounter_df = spark.table("db_healthcare_kl.gold.encounter_summaries")

cohort_df = spark.table("db_healthcare_kl.gold.patient_cohorts")

readmission_df = spark.table("db_healthcare_kl.gold.readmission_within_30days")

person_df = spark.table("db_healthcare_kl.omop.person_data")

visit_df = spark.table("db_healthcare_kl.omop.visit_occurrence")

condition_df = spark.table("db_healthcare_kl.omop.condition_occurrence")

drug_df = spark.table("db_healthcare_kl.omop.drug_exposure")

In [0]:
person_df.printSchema()

visit_df.printSchema()

condition_df.printSchema()

drug_df.printSchema()

In [0]:
person_features = (
    person_df
    .withColumn("date_of_birth", to_date("date_of_birth"))
    .withColumn(
        "age",
        year(current_date()) - year(col("date_of_birth"))
    )
    .select(
        "person_id",
        "age",
        "gender_concept_id"
    )
)

display(person_features)

In [0]:
diagnosis_features = (
    condition_df
    .groupBy("person_id")
    .agg(
        count("*").alias("diagnosis_count")
    )
)

display(diagnosis_features)

In [0]:
medication_features = (
    drug_df
    .groupBy("person_id")
    .agg(
        count("*").alias("medication_count")
    )
)

display(medication_features)

In [0]:
visit_features = (
    visit_df
    .groupBy("person_id")
    .agg(
        count("*").alias("previous_admissions")
    )
)

display(visit_features)

In [0]:
visit_type_features = (
    visit_df
    .groupBy("person_id")
    .agg(
        first("visit_type").alias("visit_type")
    )
)

display(visit_type_features)

In [0]:
diagnosis_name_features = (
    condition_df
    .groupBy("person_id")
    .agg(
        first("condition_name").alias("primary_diagnosis")
    )
)

display(diagnosis_name_features)

In [0]:
ml_df = (
    encounter_df

    .join(person_features, "person_id", "left")

    .join(diagnosis_features, "person_id", "left")

    .join(medication_features, "person_id", "left")

    .join(visit_features, "person_id", "left")

    .join(visit_type_features, "person_id", "left")

    .join(diagnosis_name_features, "person_id", "left")

    .join(cohort_df, "person_id", "left")

    .join(readmission_df, "person_id", "left")
)

In [0]:
ml_df = ml_df.fillna({

    "age":0,

    "gender_concept_id":0,

    "diagnosis_count":0,

    "medication_count":0,

    "previous_admissions":0,

    "visit_type":"Unknown",

    "primary_diagnosis":"Unknown",

    "cohort_name":"Unknown",

    "readmission_30day_flag":0
})

In [0]:
display(ml_df)

print(ml_df.count())

ml_df.printSchema()

In [0]:
(
    ml_df.write
    .mode("overwrite")
    .saveAsTable(
        "db_healthcare_kl.gold.ml_readmission_features"
    )
)

In [0]:
ml_data = spark.table("db_healthcare_kl.gold.ml_readmission_features")

In [0]:
df = ml_data.select(
    "total_visits",
    "max_duration_days",
    "avg_duration_days",
    "age",
    "gender_concept_id",
    "diagnosis_count",
    "medication_count",
    "previous_admissions",
    "visit_type",
    "primary_diagnosis",
    "Cohort_name",
    "readmission_30day_flag"
)

In [0]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

In [0]:
df = df.na.drop(
    subset=["max_duration_days", "avg_duration_days"]
)

In [0]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

Now we will convert visit type, primary diagnosis, cohort name from string to number as ML couldnt understand string first we will convert string to indexing then we have to convert that into one hot otherwise model will become consume and assume higher index as higher value one hot cannot be done directly from string so we have to do indexing first

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [0]:
visit_indexer = StringIndexer(
    inputCol="visit_type",
    outputCol="visit_type_index",
    handleInvalid="keep"
)

visit_model = visit_indexer.fit(df)

df = visit_model.transform(df)

In [0]:
display(df.select("visit_type", "visit_type_index"))

In [0]:
visit_encoder = OneHotEncoder(
    inputCols=["visit_type_index"],
    outputCols=["visit_type_vec"]
)

visit_encoder_model = visit_encoder.fit(df)

df = visit_encoder_model.transform(df)

In [0]:
display(df.select("visit_type", "visit_type_index", "visit_type_vec"))

In [0]:
diagnosis_indexer = StringIndexer(
    inputCol="primary_diagnosis",
    outputCol="primary_diagnosis_index",
    handleInvalid="keep"
)

diagnosis_model = diagnosis_indexer.fit(df)

df = diagnosis_model.transform(df)

In [0]:
display(df.select("primary_diagnosis", "primary_diagnosis_index"))

In [0]:
diagnosis_encoder = OneHotEncoder(
    inputCols=["primary_diagnosis_index"],
    outputCols=["primary_diagnosis_vec"]
)

diagnosis_encoder_model = diagnosis_encoder.fit(df)

df = diagnosis_encoder_model.transform(df)

In [0]:
display(df.select(
    "primary_diagnosis",
    "primary_diagnosis_index",
    "primary_diagnosis_vec"
))

In [0]:
cohort_indexer = StringIndexer(
    inputCol="Cohort_name",
    outputCol="cohort_index",
    handleInvalid="keep"
)

cohort_model = cohort_indexer.fit(df)

df = cohort_model.transform(df)

In [0]:
display(df.select("Cohort_name", "cohort_index"))

In [0]:
cohort_encoder = OneHotEncoder(
    inputCols=["cohort_index"],
    outputCols=["cohort_vec"]
)

cohort_encoder_model = cohort_encoder.fit(df)

df = cohort_encoder_model.transform(df)

In [0]:
display(df.select(
    "Cohort_name",
    "cohort_index",
    "cohort_vec"
))

In [0]:
df.printSchema()

now we will create a feature vector because machine learning algorithms cannot train on multiple separate columns; they require all input features to be combined into a single numerical vector for efficient processing and prediction.

In [0]:
from pyspark.ml.feature import VectorAssembler

In [0]:
assembler = VectorAssembler(
    inputCols=[
        "total_visits",
        "max_duration_days",
        "avg_duration_days",
        "age",
        "gender_concept_id",
        "diagnosis_count",
        "medication_count",
        "previous_admissions",
        "visit_type_vec",
        "primary_diagnosis_vec",
        "cohort_vec"
    ],
    outputCol="features",
    handleInvalid="keep"
)

In [0]:
df = assembler.transform(df)

In [0]:
df.select("features").show(5, truncate=False)

In [0]:
from pyspark.sql.functions import col

final_df = df.select(
    "features",
    col("readmission_30day_flag").alias("label")
)

In [0]:
final_df.show(5, truncate=False)

In [0]:
final_df.printSchema()

In [0]:
pdf = final_df.toPandas()

In [0]:
import pandas as pd

X = pd.DataFrame(
    pdf["features"].tolist()
)

y = pdf["label"]

In [0]:
import numpy as np

X = pd.DataFrame(np.vstack(pdf["features"].apply(lambda x: x.toArray())))

pd.DataFrame() on a Series of arrays creates a single object column, whereas np.vstack() combines all arrays into a 2D numeric matrix, allowing Pandas to create separate numeric feature columns that XGBoost can process.

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [0]:
# %pip install xgboost

In [0]:
# dbutils.library.restartPython()

In [0]:
from xgboost import XGBClassifier

In [0]:
print(X.dtypes)

In [0]:
model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

In [0]:
model.fit(
    X_train,
    y_train
)

In [0]:
# Predicted class labels (0 or 1)
y_pred = model.predict(X_test)

# Predicted probabilities for class 1
y_prob = model.predict_proba(X_test)[:, 1]

In [0]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [0]:
accuracy = accuracy_score(
    y_test,
       y_pred
)

In [0]:
precision = precision_score(y_test, y_pred)

In [0]:
recall = recall_score(
    y_test,
       y_pred
)

In [0]:
f1 = f1_score(
    y_test,
       y_pred
)

In [0]:
roc = roc_auc_score(
    y_test,
       y_prob
)

In [0]:
cm = confusion_matrix(
    y_test,
       y_pred
)

print(cm)

In [0]:
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc)
print("Confusion Matrix:\n", cm)